In [2]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
pasta_ibcbr = BASE_DIR / "data" / "bronze" / "ibcbr"

arquivos = list(pasta_ibcbr.glob("*.json"))
assert len(arquivos) > 0, "Nenhum arquivo encontrado em data/bronze/ibcbr/"

In [3]:
df_raw = pd.read_json(arquivos[0])
print(f"Arquivo: {arquivos[0].name} | Linhas: {len(df_raw)}")
df_raw.head()

Arquivo: ibcbr_2026-09-06.json | Linhas: 58


,data,valor
0,01/09/2021,96.95313
1,01/10/2021,97.47791
2,01/11/2021,99.05295
3,01/12/2021,99.36759
4,01/01/2022,97.41759


In [4]:
df = df_raw.copy()
df = df.rename(columns={"valor": "indice_ibcbr"})
df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y").dt.to_period("M").dt.to_timestamp()
df["indice_ibcbr"] = pd.to_numeric(df["indice_ibcbr"], errors="coerce")
df = df.sort_values("data").reset_index(drop=True)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   data          58 non-null     datetime64[us]
 1   indice_ibcbr  58 non-null     float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 1.0 KB


In [5]:
assert not df["data"].duplicated().any(), "Erro: Meses duplicados no IBC-Br!"
assert (df["indice_ibcbr"] > 0).all(), "Erro: Índice do IBC-Br com valor menor ou igual a zero!"

display(df["indice_ibcbr"].describe())

count     58.000000
mean     104.743472
std        4.171380
min       96.953130
25%      101.221802
50%      104.674005
75%      108.701543
max      110.931020
Name: indice_ibcbr, dtype: float64

In [6]:
df["crescimento_mom_pct"] = (df["indice_ibcbr"].pct_change() * 100).round(2)
display(df.tail(6))

,data,indice_ibcbr,crescimento_mom_pct
52,2026-01-01,110.07661,0.56
53,2026-02-01,110.77581,0.64
54,2026-03-01,110.60765,-0.15
55,2026-04-01,110.90086,0.27
56,2026-05-01,110.93102,0.03
57,2026-06-01,110.22154,-0.64


In [7]:
pasta_silver = BASE_DIR / "data" / "silver"
pasta_silver.mkdir(parents=True, exist_ok=True)

# Gravação do teste
caminho_teste_parquet = pasta_silver / "ibcbr_silver_test.parquet"
df.to_parquet(caminho_teste_parquet, index=False)

print("escrita Parquet concluído")
print(f"Arquivo salvo em: {caminho_teste_parquet.relative_to(BASE_DIR)}")

escrita Parquet concluído
Arquivo salvo em: data/silver/ibcbr_silver_test.parquet
